# Federated Learning Model Training - UCI HAR Dataset
## Federated Learning Approach with Multiple Clients

This notebook trains a federated learning model for Human Activity Recognition.

**Target Accuracy:** >75%

**Dataset:** UCI HAR Dataset (561 features, 6 activity classes)

**Approach:** Simulated federated learning with multiple clients (based on subject IDs)


## 1. Setup and Installation


In [ ]:
# Install required packages
%pip install numpy pandas matplotlib seaborn scikit-learn tensorflow joblib -q


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
import os
import json
import joblib
from datetime import datetime
from copy import deepcopy

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")


## 2. Mount Google Drive and Upload Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Set the path to your dataset
DATASET_PATH = '/content/drive/MyDrive/UCI_HAR_Dataset'

print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {os.path.exists(DATASET_PATH)}")


## 3. Load and Prepare Data


In [ ]:
def load_uci_har_dataset_with_subjects(dataset_path):
    """
    Load UCI HAR Dataset including subject information for federated learning.
    
    Returns:
        X_train, y_train, subjects_train, X_test, y_test, subjects_test, feature_names, activity_labels
    """
    # Load training data
    X_train = np.loadtxt(os.path.join(dataset_path, 'train', 'X_train.txt'))
    y_train = np.loadtxt(os.path.join(dataset_path, 'train', 'y_train.txt'))
    subjects_train = np.loadtxt(os.path.join(dataset_path, 'train', 'subject_train.txt'))
    
    # Load test data
    X_test = np.loadtxt(os.path.join(dataset_path, 'test', 'X_test.txt'))
    y_test = np.loadtxt(os.path.join(dataset_path, 'test', 'y_test.txt'))
    subjects_test = np.loadtxt(os.path.join(dataset_path, 'test', 'subject_test.txt'))
    
    # Load feature names
    with open(os.path.join(dataset_path, 'features.txt'), 'r') as f:
        feature_names = [line.strip().split()[1] for line in f.readlines()]
    
    # Load activity labels
    with open(os.path.join(dataset_path, 'activity_labels.txt'), 'r') as f:
        activity_labels = [line.strip().split()[1] for line in f.readlines()]
    
    # Convert labels to zero-indexed (original is 1-6, convert to 0-5)
    y_train = y_train - 1
    y_test = y_test - 1
    
    return X_train, y_train, subjects_train, X_test, y_test, subjects_test, feature_names, activity_labels

# Load the dataset
X_train, y_train, subjects_train, X_test, y_test, subjects_test, feature_names, activity_labels = load_uci_har_dataset_with_subjects(DATASET_PATH)

print("Dataset loaded successfully!")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Number of classes: {len(activity_labels)}")
print(f"Activity labels: {activity_labels}")
print(f"Number of unique training subjects: {len(np.unique(subjects_train))}")
print(f"Number of unique test subjects: {len(np.unique(subjects_test))}")


In [ ]:
# Analyze subject distribution
print("\n=== Subject Distribution ===")
print(f"\nTraining subjects: {sorted(np.unique(subjects_train).astype(int))}")
print(f"Test subjects: {sorted(np.unique(subjects_test).astype(int))}")

# Count samples per subject in training
unique_subjects_train = np.unique(subjects_train)
print(f"\n=== Training Samples per Subject ===")
for subject in sorted(unique_subjects_train)[:10]:  # Show first 10
    count = np.sum(subjects_train == subject)
    print(f"  Subject {int(subject)}: {count} samples")


## 4. Data Preprocessing


In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data standardized successfully!")
print(f"X_train_scaled mean: {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std: {X_train_scaled.std():.6f}")

# Convert labels to categorical
num_classes = len(activity_labels)
y_train_categorical = to_categorical(y_train, num_classes)
y_test_categorical = to_categorical(y_test, num_classes)

print(f"\nLabels converted to categorical format")
print(f"y_train_categorical shape: {y_train_categorical.shape}")
print(f"y_test_categorical shape: {y_test_categorical.shape}")


## 5. Prepare Federated Learning Clients

We'll simulate federated learning by treating each subject as a separate client. This mimics the real-world scenario where data is distributed across multiple devices.


In [ ]:
def create_federated_clients(X, y, subjects):
    """
    Create federated clients where each subject is a client.
    
    Returns:
        Dictionary of clients with their data
    """
    clients = {}
    unique_subjects = np.unique(subjects)
    
    for subject in unique_subjects:
        # Get indices for this subject
        indices = np.where(subjects == subject)[0]
        
        # Create client data
        client_id = f"client_{int(subject)}"
        clients[client_id] = {
            'X': X[indices],
            'y': y[indices],
            'subject_id': int(subject),
            'num_samples': len(indices)
        }
    
    return clients

# Create federated clients from training data
federated_clients = create_federated_clients(X_train_scaled, y_train_categorical, subjects_train)

print(f"Created {len(federated_clients)} federated clients")
print(f"\nClient statistics:")
sample_counts = [client['num_samples'] for client in federated_clients.values()]
print(f"  Min samples per client: {min(sample_counts)}")
print(f"  Max samples per client: {max(sample_counts)}")
print(f"  Mean samples per client: {np.mean(sample_counts):.1f}")
print(f"  Median samples per client: {np.median(sample_counts):.1f}")


In [ ]:
# Visualize client distribution
plt.figure(figsize=(14, 5))

# Plot samples per client
client_ids = list(federated_clients.keys())
client_samples = [federated_clients[cid]['num_samples'] for cid in client_ids]
client_labels = [f"C{federated_clients[cid]['subject_id']}" for cid in client_ids]

plt.bar(client_labels, client_samples, color='steelblue')
plt.title('Number of Samples per Federated Client', fontsize=14, fontweight='bold')
plt.xlabel('Client ID')
plt.ylabel('Number of Samples')
plt.axhline(y=np.mean(client_samples), color='r', linestyle='--', linewidth=2, label=f'Mean: {np.mean(client_samples):.1f}')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 6. Build Federated Learning Model

We'll use the SAME architecture as the Deep Learning model to ensure fair comparison.


In [ ]:
def create_federated_model(input_shape, num_classes):
    """
    Create the same neural network architecture as deep learning model.
    This ensures fair comparison between FL and DL.
    
    Args:
        input_shape: Shape of input features (561,)
        num_classes: Number of output classes (6)
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential([
        # Input layer
        layers.Input(shape=input_shape),
        
        # First hidden layer
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        # Second hidden layer
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        
        # Third hidden layer
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Fourth hidden layer
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ])
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Create global model
input_shape = (X_train_scaled.shape[1],)
global_model = create_federated_model(input_shape, num_classes)

# Display model architecture
global_model.summary()


## 7. Federated Learning Training Functions


In [ ]:
def train_client_model(global_weights, client_data, epochs=5, batch_size=32, verbose=0):
    """
    Train a local model on client data.
    
    Args:
        global_weights: Current global model weights
        client_data: Dictionary with 'X' and 'y' for this client
        epochs: Number of local training epochs
        batch_size: Batch size for training
        verbose: Verbosity level
    
    Returns:
        Updated weights and training history
    """
    # Create local model
    local_model = create_federated_model(input_shape, num_classes)
    
    # Set global weights
    local_model.set_weights(global_weights)
    
    # Train locally
    history = local_model.fit(
        client_data['X'],
        client_data['y'],
        batch_size=batch_size,
        epochs=epochs,
        verbose=verbose,
        validation_split=0.0  # No validation in local training
    )
    
    # Return updated weights
    return local_model.get_weights(), history

def federated_averaging(client_weights_list, client_sample_counts):
    """
    Perform federated averaging (FedAvg) to aggregate client weights.
    
    Args:
        client_weights_list: List of client model weights
        client_sample_counts: List of number of samples per client
    
    Returns:
        Averaged weights
    """
    # Calculate total samples
    total_samples = sum(client_sample_counts)
    
    # Initialize averaged weights
    averaged_weights = []
    
    # For each layer's weights
    for layer_idx in range(len(client_weights_list[0])):
        # Weighted average based on number of samples
        layer_avg = np.zeros_like(client_weights_list[0][layer_idx])
        
        for client_idx, client_weights in enumerate(client_weights_list):
            weight = client_sample_counts[client_idx] / total_samples
            layer_avg += weight * client_weights[layer_idx]
        
        averaged_weights.append(layer_avg)
    
    return averaged_weights

print("Federated learning functions defined successfully!")


In [ ]:
# Federated learning hyperparameters
NUM_ROUNDS = 50  # Number of federated rounds
LOCAL_EPOCHS = 5  # Number of epochs per client per round
LOCAL_BATCH_SIZE = 32
CLIENTS_PER_ROUND = 10  # Number of clients to sample per round (for efficiency)

# Track training history
fl_history = {
    'round': [],
    'train_loss': [],
    'train_accuracy': [],
    'test_loss': [],
    'test_accuracy': []
}

print(f"Starting Federated Learning Training")
print(f"  Rounds: {NUM_ROUNDS}")
print(f"  Local Epochs: {LOCAL_EPOCHS}")
print(f"  Clients per Round: {CLIENTS_PER_ROUND}")
print(f"  Total Clients: {len(federated_clients)}")
print()


In [ ]:
# Main federated learning loop
client_ids = list(federated_clients.keys())

for round_num in range(1, NUM_ROUNDS + 1):
    print(f"\n{'='*60}")
    print(f"Federated Learning Round {round_num}/{NUM_ROUNDS}")
    print(f"{'='*60}")
    
    # Sample clients for this round
    sampled_client_ids = np.random.choice(
        client_ids, 
        size=min(CLIENTS_PER_ROUND, len(client_ids)), 
        replace=False
    )
    
    # Get current global weights
    global_weights = global_model.get_weights()
    
    # Train on each selected client
    client_weights_list = []
    client_sample_counts = []
    
    for client_id in sampled_client_ids:
        client_data = federated_clients[client_id]
        
        # Train client model
        updated_weights, _ = train_client_model(
            global_weights,
            client_data,
            epochs=LOCAL_EPOCHS,
            batch_size=LOCAL_BATCH_SIZE,
            verbose=0
        )
        
        client_weights_list.append(updated_weights)
        client_sample_counts.append(client_data['num_samples'])
    
    # Aggregate weights using federated averaging
    aggregated_weights = federated_averaging(client_weights_list, client_sample_counts)
    
    # Update global model
    global_model.set_weights(aggregated_weights)
    
    # Evaluate on training set (sample for efficiency)
    train_indices = np.random.choice(len(X_train_scaled), min(1000, len(X_train_scaled)), replace=False)
    train_loss, train_acc = global_model.evaluate(
        X_train_scaled[train_indices],
        y_train_categorical[train_indices],
        verbose=0
    )
    
    # Evaluate on test set
    test_loss, test_acc = global_model.evaluate(
        X_test_scaled,
        y_test_categorical,
        verbose=0
    )
    
    # Store history
    fl_history['round'].append(round_num)
    fl_history['train_loss'].append(train_loss)
    fl_history['train_accuracy'].append(train_acc)
    fl_history['test_loss'].append(test_loss)
    fl_history['test_accuracy'].append(test_acc)
    
    # Print results
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"  Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc*100:.2f}%")
    
    # Early stopping if target is achieved
    if test_acc >= 0.75 and round_num >= 10:
        print(f"\n✓ Target accuracy of 75% achieved at round {round_num}!")

print(f"\n{'='*60}")
print("Federated Learning Training Completed!")
print(f"{'='*60}")


## 9. Visualize Training History


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(fl_history['round'], fl_history['train_accuracy'], label='Train Accuracy', linewidth=2, marker='o')
axes[0].plot(fl_history['round'], fl_history['test_accuracy'], label='Test Accuracy', linewidth=2, marker='s')
axes[0].axhline(y=0.75, color='r', linestyle='--', linewidth=2, alpha=0.7, label='Target (75%)')
axes[0].set_title('Federated Learning - Accuracy Over Rounds', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Federated Round')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(fl_history['round'], fl_history['train_loss'], label='Train Loss', linewidth=2, marker='o')
axes[1].plot(fl_history['round'], fl_history['test_loss'], label='Test Loss', linewidth=2, marker='s')
axes[1].set_title('Federated Learning - Loss Over Rounds', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Federated Round')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fl_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

# Print best results
best_test_acc = max(fl_history['test_accuracy'])
best_round = fl_history['test_accuracy'].index(best_test_acc) + 1
print(f"\nBest Test Accuracy: {best_test_acc*100:.2f}% (Round {best_round})")


## 10. Final Evaluation


In [ ]:
# Final evaluation on test set
print("Final Evaluation on Test Set...\n")

test_loss, test_accuracy = global_model.evaluate(X_test_scaled, y_test_categorical, verbose=0)

print(f"\n{'='*50}")
print(f"FEDERATED LEARNING MODEL - TEST RESULTS")
print(f"{'='*50}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"{'='*50}")

# Check if target accuracy is met
target_accuracy = 0.75
if test_accuracy >= target_accuracy:
    print(f"\n✓ Target accuracy of {target_accuracy*100}% ACHIEVED!")
else:
    print(f"\n✗ Target accuracy of {target_accuracy*100}% NOT achieved.")
    print(f"  Difference: {(target_accuracy - test_accuracy)*100:.2f}%")


In [ ]:
# Make predictions
y_pred_probs = global_model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Print detailed classification report
print("\n=== Classification Report ===")
print(classification_report(
    y_test,
    y_pred,
    target_names=activity_labels,
    digits=4
))


## 11. Confusion Matrix Visualization


In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=activity_labels, yticklabels=activity_labels,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix (Raw Counts)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized confusion matrix
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=activity_labels, yticklabels=activity_labels,
            ax=axes[1], cbar_kws={'label': 'Proportion'})
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('fl_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


## 12. Per-Class Performance Analysis


In [ ]:
# Calculate per-class accuracy
class_accuracies = cm.diagonal() / cm.sum(axis=1)

# Create DataFrame
performance_df = pd.DataFrame({
    'Activity': activity_labels,
    'Accuracy': class_accuracies * 100,
    'Samples': cm.sum(axis=1)
})
performance_df = performance_df.sort_values('Accuracy', ascending=False)

print("\n=== Per-Class Performance ===")
print(performance_df.to_string(index=False))

# Visualize per-class accuracy
plt.figure(figsize=(12, 6))
bars = plt.bar(performance_df['Activity'], performance_df['Accuracy'], 
               color=plt.cm.plasma(performance_df['Accuracy']/100))
plt.axhline(y=test_accuracy*100, color='r', linestyle='--', linewidth=2, label=f'Overall Accuracy: {test_accuracy*100:.2f}%')
plt.title('Per-Class Accuracy - Federated Learning Model', fontsize=14, fontweight='bold')
plt.xlabel('Activity')
plt.ylabel('Accuracy (%)')
plt.ylim([0, 105])
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('fl_per_class_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()


## 13. Save the Federated Model


In [ ]:
# Save the federated model in multiple formats

# 1. Save as H5 format
global_model.save('federated_learning_model.h5')
print("Model saved as 'federated_learning_model.h5'")

# 2. Save as SavedModel format
global_model.save('federated_learning_model_savedmodel', save_format='tf')
print("Model saved as 'federated_learning_model_savedmodel'")

# 3. Save as TFLite format (for mobile deployment)
converter = tf.lite.TFLiteConverter.from_keras_model(global_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('federated_learning_model.tflite', 'wb') as f:
    f.write(tflite_model)
print("Model saved as 'federated_learning_model.tflite'")

# 4. Save the scaler
joblib.dump(scaler, 'scaler_fl.pkl')
print("Scaler saved as 'scaler_fl.pkl'")

# Get model file sizes
h5_size = os.path.getsize('federated_learning_model.h5') / 1024  # KB
tflite_size = os.path.getsize('federated_learning_model.tflite') / 1024  # KB

print(f"\nModel Sizes:")
print(f"  H5 format: {h5_size:.2f} KB")
print(f"  TFLite format: {tflite_size:.2f} KB")


## 14. Model Information Summary


In [ ]:
# Create model information dictionary
model_info = {
    'model_type': 'Federated Learning',
    'model_name': 'UCI_HAR_FL_Model',
    'framework': 'TensorFlow/Keras',
    'date_trained': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': 'UCI HAR Dataset',
    'input_shape': list(input_shape),
    'num_classes': num_classes,
    'class_labels': activity_labels,
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'num_features': int(X_train.shape[1]),
    'model_architecture': {
        'total_layers': len(global_model.layers),
        'total_parameters': int(global_model.count_params()),
        'trainable_parameters': int(sum([tf.size(w).numpy() for w in global_model.trainable_weights]))
    },
    'federated_config': {
        'num_clients': len(federated_clients),
        'num_rounds': NUM_ROUNDS,
        'local_epochs': LOCAL_EPOCHS,
        'local_batch_size': LOCAL_BATCH_SIZE,
        'clients_per_round': CLIENTS_PER_ROUND,
        'aggregation_method': 'FedAvg (Federated Averaging)'
    },
    'performance_metrics': {
        'test_accuracy': float(test_accuracy),
        'test_loss': float(test_loss),
        'best_test_accuracy': float(best_test_acc),
        'best_round': int(best_round),
        'target_accuracy_met': bool(test_accuracy >= 0.75)
    },
    'model_sizes': {
        'h5_kb': float(h5_size),
        'tflite_kb': float(tflite_size)
    }
}

# Save model info as JSON
with open('federated_learning_model_info.json', 'w') as f:
    json.dump(model_info, f, indent=4)

print("\n=== MODEL INFORMATION ===")
print(json.dumps(model_info, indent=2))
print("\nModel information saved as 'federated_learning_model_info.json'")


## 15. Download Files to Local Machine


In [ ]:
# Download all important files
from google.colab import files

files_to_download = [
    'federated_learning_model.h5',
    'federated_learning_model.tflite',
    'federated_learning_model_info.json',
    'scaler_fl.pkl',
    'fl_training_history.png',
    'fl_confusion_matrix.png',
    'fl_per_class_accuracy.png'
]

print("Downloading files...")
for file in files_to_download:
    if os.path.exists(file):
        files.download(file)
        print(f"  ✓ {file}")
    else:
        print(f"  ✗ {file} not found")

print("\nDownload complete!")


## 16. Test Model Inference


In [ ]:
# Test inference on random samples
num_samples = 5
random_indices = np.random.choice(len(X_test_scaled), num_samples, replace=False)

print("=== Testing Model Inference ===")
print("\nRandom Sample Predictions:\n")

for i, idx in enumerate(random_indices, 1):
    sample = X_test_scaled[idx:idx+1]
    true_label = int(y_test[idx])
    
    # Make prediction
    prediction_probs = global_model.predict(sample, verbose=0)[0]
    predicted_label = np.argmax(prediction_probs)
    confidence = prediction_probs[predicted_label] * 100
    
    status = "✓" if predicted_label == true_label else "✗"
    
    print(f"Sample {i}:")
    print(f"  True Activity: {activity_labels[true_label]}")
    print(f"  Predicted Activity: {activity_labels[predicted_label]}")
    print(f"  Confidence: {confidence:.2f}%")
    print(f"  Status: {status}")
    print()


## Summary

This notebook successfully trained a federated learning model for human activity recognition using the UCI HAR Dataset.

**Key Points:**
- Model architecture: Same as Deep Learning model (for fair comparison)
- Input: 561 features from accelerometer and gyroscope data
- Output: 6 activity classes
- Target accuracy: >75%
- Federated approach: Each subject as a separate client
- Aggregation: Federated Averaging (FedAvg)
- Model saved in multiple formats for deployment

**Federated Learning Benefits:**
1. Privacy-preserving: Data stays on clients
2. Distributed training: No need to centralize data
3. Scalable: Can handle many clients

**Next Steps:**
1. Use the saved model for API integration
2. Deploy TFLite model to mobile devices
3. Compare performance with Deep Learning model
